# Laboratorio de regresión logística

|                |   |
:----------------|---|
| **Nombre**     |Sofia Anaya Palafox   |
| **Fecha**      |15 marzo 2026   |
| **Expediente** |738594   |

In machine learning, Support Vector Machines (SVM) are supervised learning models with associated learning algorithms that analyze data used for classification and regression analysis. It is mostly used in classification problems. In this algorithm, each data item is plotted as a point in p-dimensional space (where p is the number of features), with the value of each feature being the value of a particular coordinate. Then, classification is performed by finding the hyper-plane that best differentiates the two classes (or more if we have a multi class problem):

$$ f(x) = w^T \varphi(x) + b $$

where $\varphi: X \rightarrow F $ is a function that makes each input point $x$ correspond to a point in F, where F is a Hilbert space.

In addition to performing linear classification, SVMs can efficiently perform a non-linear classification, implicitly mapping their inputs into high-dimensional feature spaces (more specifically using the kernel trick, like the RBF funcion).

[1]

OLS utilizes the squared residuals to fit the parameters. Large residuals caused by outliers may worsen the accuracy significantly.

Support Vectors use piecewise linear functions to counter this, in which a hyperparameter  $\epsilon$ called the margin lets errors that are less or equal to it be 0, and error larger than it be $e - \epsilon$.

The problem to solve is:

\begin{split}
        \min_{w, b, \xi, \xi^*} \mathcal{P}_\epsilon(w, b, \xi) &= \frac{1}{2} w^T w + c \sum_{k=1}^{N} \xi_k \\
        \text{s.t. } & y_k [w^T \varphi(x_k) - b] \geq 1- \xi_k,\ \ k = 1, ..., N \\
        & \xi_k \geq 0,\ \ k = 1, ..., N
\end{split}


The most important question that arises when using a SVM is how to choose the correct hyperplane. Consider the following scenarios:

### Scenario 1

In this scenario there are three hyperplanes called A, B, and C. Now, the problem is to identify the hyperplane which best differentiates the stars and the circles.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_21-2.png" alt="what image shows"></center>

In this case, hyperplane B separates the stars and the circle betters, hence it is the correct hyperplane.


### Scenario 2

Now take another scenario where all three hyperplanes are segregating classes well. The question that arises is how to choose the best hyperplane in this situation.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_4-2.png" alt="what image shows"></center>

In such scenarios, we calculate the margin (which is the distance between nearest data point and the hyperplane). The hyperplane with the largest margin will be considered as the correct hyperplane to classify the dataset.

Here C has the largest margin. Hence, it is considered as the best hyperplane.


### Kernels
Knowing
$$ w = \sum_{k=1}^{N} \alpha_k y_k \varphi(x_k) $$

And
$$ y_{pred} = w^T \varphi(x) + b $$

Then
$$ y_{pred} = (\sum_{k=1}^{N} \alpha_k y_k \varphi(x_k))^T \varphi(x) + b $$

Where $\varphi$ is a function that makes each input in $x$ correspond to a point in $F$ (a Hilbert space). This can be seen as processing and transforming the input featuers to keep the model's convexity. [2]

This also allows us to transform the inputs into another space where they might be more easily classified.

<center><img src=https://miro.medium.com/max/838/1*gXvhD4IomaC9Jb37tzDUVg.png alt="what image shows"></center>

## ROC and AUC

A ROC (Receiver Operating Characteristic) is a graph that shows how the classification model performs at the classification thresholds.

ROC curves typically feature true positive rate on the Y axis, and false positive rate on the X axis. This means that the top left corner of the plot is the “ideal” point - a false positive rate of zero, and a true positive rate of one. This is not very realistic, but it does mean that a larger area under the curve (AUC) is usually better. [3]

True Positive Rate is a synonym for Recall and defined as:
$$ TPR = \frac{TP}{TP + FN} $$

False Positive Rate is a synonym for Specificity and defined as:

$$ FPR = \frac{FP}{FP + TN} $$

ROC curves are typically used in binary classification to study the output of a classifier. In order to extend ROC curve and ROC area to multi-label classification, it is necessary to binarize the output. One ROC curve can be drawn per label, but one can also draw a ROC curve by considering each element of the label indicator matrix as a binary prediction (micro-averaging).

E.g. If you lower a classification threshold, more items would be classified as positive, increasing False Positives and True Positives.

AUC stands for Area under the ROC.

## Ejercicio 1

- Utiliza el dataset `Iris`, modela con SVC y haz Cross-Validation de diferentes kernels ('linear', 'poly', 'rbf', 'sigmoid').
- Modela con LogisticRegression.
- El método de Cross-Validation es K-Folds con $k=10$.
- Utiliza el AUC como métrico de Cross-Validation.
- Compara resultados.

In [1]:
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

iris = load_iris()
X, y = iris.data, iris.target

models = {
    'SVC_linear': SVC(kernel='linear', probability=True, random_state=42),
    'SVC_poly': SVC(kernel='poly', probability=True, random_state=42),
    'SVC_rbf': SVC(kernel='rbf', probability=True, random_state=42),
    'SVC_sigmoid': SVC(kernel='sigmoid', probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=200, random_state=42)
}

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = {}

print("Performing Cross-Validation...")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kf, scoring='roc_auc_ovr_weighted', n_jobs=-1)
    results[name] = {
        'mean_auc': np.mean(scores),
        'std_auc': np.std(scores),
        'scores': scores
    }
    print(f"  {name}: Mean AUC = {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

results_df = pd.DataFrame({
    'Model': [name for name in results.keys()],
    'Mean AUC': [data['mean_auc'] for data in results.values()],
    'Std AUC': [data['std_auc'] for data in results.values()]
})

print("\nComparison of Models (AUC Scores):")
display(results_df.sort_values(by='Mean AUC', ascending=False))


Performing Cross-Validation...
  SVC_linear: Mean AUC = 0.9960 (+/- 0.0085)
  SVC_poly: Mean AUC = 0.9960 (+/- 0.0085)
  SVC_rbf: Mean AUC = 0.9973 (+/- 0.0080)
  SVC_sigmoid: Mean AUC = 0.7467 (+/- 0.0490)
  LogisticRegression: Mean AUC = 0.9960 (+/- 0.0085)

Comparison of Models (AUC Scores):


,Model,Mean AUC,Std AUC
2,SVC_rbf,0.997333,0.008000
0,SVC_linear,0.996000,0.008537
1,SVC_poly,0.996000,0.008537
4,LogisticRegression,0.996000,0.008537
3,SVC_sigmoid,0.746667,0.048990


## Ejercicio 2
- Repite el ejercicio 1 con el dataset `Default`. Utiliza `default` como target.

In [6]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np

default_df = pd.read_csv('/content/Default (1).csv')


display(default_df.head())

default_df.info()


,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138950
3,No,No,529.250605,35704.493940
4,No,No,785.655883,38463.495880


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   default  10000 non-null  object 
 1   student  10000 non-null  object 
 2   balance  10000 non-null  float64
 3   income   10000 non-null  float64
dtypes: float64(2), object(2)
memory usage: 312.6+ KB


In [8]:

default_df['student'] = default_df['student'].map({'Yes': 1, 'No': 0})

y_default = default_df['default'].map({'Yes': 1, 'No': 0})

X_default = default_df[['income', 'balance', 'student']]

scaler = StandardScaler()
X_default_scaled = scaler.fit_transform(X_default)

print(pd.DataFrame(X_default_scaled, columns=X_default.columns).head())


print(y_default.head())


     income   balance  student
0  0.813187 -0.218835      NaN
1 -1.605496 -0.037616      NaN
2 -0.131212  0.492410      NaN
3  0.164031 -0.632893      NaN
4  0.370915 -0.102791      NaN
0    0
1    0
2    0
3    0
4    0
Name: default, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [4]:
models_default = {
    'SVC_linear': SVC(kernel='linear', probability=True, random_state=42),
    'SVC_poly': SVC(kernel='poly', probability=True, random_state=42),
    'SVC_rbf': SVC(kernel='rbf', probability=True, random_state=42),
    'SVC_sigmoid': SVC(kernel='sigmoid', probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=200, random_state=42, solver='liblinear') # 'liblinear' for small datasets and binary target
}

kf_default = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results_default = {}

print("Performing Cross-Validation for Default dataset...")
for name, model in models_default.items():
    scores_default = cross_val_score(model, X_default_scaled, y_default, cv=kf_default, scoring='roc_auc', n_jobs=-1)
    results_default[name] = {
        'mean_auc': np.mean(scores_default),
        'std_auc': np.std(scores_default),
        'scores': scores_default
    }
    print(f"  {name}: Mean AUC = {np.mean(scores_default):.4f} (+/- {np.std(scores_default):.4f})")

results_default_df = pd.DataFrame({
    'Model': [name for name in results_default.keys()],
    'Mean AUC': [data['mean_auc'] for data in results_default.values()],
    'Std AUC': [data['std_auc'] for data in results_default.values()]
})

print("\nComparison of Models (AUC Scores) for Default dataset:")
display(results_default_df.sort_values(by='Mean AUC', ascending=False))


Performing Cross-Validation for Default dataset...
  SVC_linear: Mean AUC = 0.9235 (+/- 0.0294)
  SVC_poly: Mean AUC = 0.8754 (+/- 0.0386)
  SVC_rbf: Mean AUC = 0.8420 (+/- 0.0400)
  SVC_sigmoid: Mean AUC = 0.7293 (+/- 0.0442)
  LogisticRegression: Mean AUC = 0.9485 (+/- 0.0221)

Comparison of Models (AUC Scores) for Default dataset:


,Model,Mean AUC,Std AUC
4,LogisticRegression,0.948494,0.022119
0,SVC_linear,0.923469,0.029397
1,SVC_poly,0.875401,0.038629
2,SVC_rbf,0.842005,0.039993
3,SVC_sigmoid,0.729256,0.044172


# Addendum

Métricos disponibles para clasificación:
- ‘accuracy’
- ‘balanced_accuracy’
- ‘top_k_accuracy’
- ‘average_precision’
- ‘neg_brier_score’
- ‘f1’
- ‘f1_micro’
- ‘f1_macro’
- ‘f1_weighted’
- ‘f1_samples’
- ‘neg_log_loss’
- ‘precision’ etc.
- ‘recall’ etc.
- ‘jaccard’ etc.
- ‘roc_auc’
- ‘roc_auc_ovr’
- ‘roc_auc_ovo’
- ‘roc_auc_ovr_weighted’
- ‘roc_auc_ovo_weighted’
- ‘d2_log_loss_score’

# References

[1] Shigeo Abe.Support Vector Machines for Pattern Classification,2Ed.Springer-Verlag London,2010. ISBN978-1-84996-097-7. URLhttps://www.springer.com/gp/book/9781849960977.

[2] Johan A K Suykens, Tony Van Gestel, Jos De Brabanter, BartDe Moor, and Joos Vandewalle.Least Squares Support VectorMachines. World Scientific,2002. ISBN9789812381514. URLhttps://www.worldscientific.com/worldscibooks/10.1142/5089.

[3] Bradley, A. P. (1997). The use of the area under the ROC curve in the evaluation of machine learning algorithms. Pattern recognition, 30(7), 1145-1159. URL https://www.researchgate.net/post/how_can_I_interpret_the_ROC_curve_result